# 1. Project Initialization and Data Access

### Objective
This section initializes the project environment and establishes the paths to the raw and processed Fannie Mae loan performance data.

### Why this is necessary
Before performing data inspection, cleaning, feature engineering, or modelling, it is important to verify that the project directories and source data are correctly configured and accessible. This ensures that all subsequent analysis is reproducible and uses the intended dataset.

### What is verified
- Project directory structure
- Processed dataset directory
- Number of processed data chunks
- Availability of the source files
- DuckDB environment used for large-scale data processing

The dataset is stored as multiple compressed CSV files because of its large size. DuckDB will be used for memory-efficient querying and aggregation instead of loading the complete dataset into pandas at once.

In [2]:
# ================================================================
# CELL 1 — PROJECT CONFIGURATION
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import os
import glob
import gc

# ------------------------------------------------
# Project paths
# ------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CHUNKS_DIR = PROCESSED_DIR / "fannie_2018Q1_chunks"
FIGURES_DIR = PROJECT_DIR / "figures"
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"

# ------------------------------------------------
# Raw processed chunk files
# ------------------------------------------------

RAW_GLOB = str(
    CHUNKS_DIR / "part_*.csv.gz"
)

FILES = sorted(
    glob.glob(RAW_GLOB)
)

# ------------------------------------------------
# Basic environment information
# ------------------------------------------------

print("=" * 70)
print("PROJECT INITIALIZATION")
print("=" * 70)

print(f"Project directory : {PROJECT_DIR}")
print(f"Chunks directory  : {CHUNKS_DIR}")
print(f"Number of chunks  : {len(FILES)}")
print(f"DuckDB version    : {duckdb.__version__}")

print("\nPath checks:")
print(f"Project exists    : {PROJECT_DIR.exists()}")
print(f"Processed exists  : {PROCESSED_DIR.exists()}")
print(f"Chunks exists     : {CHUNKS_DIR.exists()}")

if FILES:
    print("\nFirst chunk:")
    print(FILES[0])

    print("\nLast chunk:")
    print(FILES[-1])
else:
    raise FileNotFoundError(
        "No processed CSV.gz chunks were found."
    )

PROJECT INITIALIZATION
Project directory : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning
Chunks directory  : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_chunks
Number of chunks  : 95
DuckDB version    : 1.5.5

Path checks:
Project exists    : True
Processed exists  : True
Chunks exists     : True

First chunk:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_chunks\part_0001.csv.gz

Last chunk:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_chunks\part_0095.csv.gz


# 2. Raw Data Schema Inspection

### Objective
The objective of this section is to inspect the structure of the raw Fannie Mae loan performance data and identify all available variables and their detected data types.

### Why this is necessary
Understanding the dataset schema is a fundamental step before data cleaning and feature engineering. It allows us to:

- verify that the expected variables are present;
- identify numerical, categorical, date-related, and behavioural fields;
- understand how the source data has been interpreted by the processing engine;
- identify variables that may require explicit type conversion;
- establish the foundation for selecting predictors and defining the early-warning target.

Only the first processed chunk is inspected at this stage because the purpose is to understand the schema rather than scan the entire dataset.

No variables are modified or removed in this step. All transformations will be performed only after the raw structure has been validated.

In [3]:
# ================================================================
# CELL 2 — RAW DATA SCHEMA INSPECTION
# ================================================================

con = duckdb.connect()

con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

schema_query = f"""
DESCRIBE
SELECT *
FROM read_csv_auto(
    '{FILES[0]}',
    header = true,
    compression = 'gzip'
)
"""

schema_df = con.execute(schema_query).fetchdf()

print("=" * 70)
print("RAW DATA SCHEMA")
print("=" * 70)

print(f"Number of columns: {len(schema_df)}")

display(schema_df)

RAW DATA SCHEMA
Number of columns: 42


,column_name,column_type,null,key,default,extra
0,loan_id,BIGINT,YES,None,None,None
1,monthly_reporting_period,VARCHAR,YES,None,None,None
2,channel,VARCHAR,YES,None,None,None
3,seller_name,VARCHAR,YES,None,None,None
4,servicer_name,VARCHAR,YES,None,None,None
5,original_interest_rate,DOUBLE,YES,None,None,None
6,current_interest_rate,DOUBLE,YES,None,None,None
7,original_upb,DOUBLE,YES,None,None,None
8,current_actual_upb,DOUBLE,YES,None,None,None
9,original_loan_term,BIGINT,YES,None,None,None


# 3. Raw Dataset Structural Validation

### Objective
The objective of this section is to verify the fundamental structure of the processed Fannie Mae loan performance dataset before performing any transformation.

### Why this is necessary
The dataset is large and stored across multiple compressed files. Before constructing the modelling dataset, we need to establish that the source data is structurally consistent across all chunks.

The following characteristics are validated:

- total number of observations;
- number of unique mortgage loans;
- chronological reporting-period coverage;
- number of distinct reporting months;
- whether each loan-month combination is unique;
- whether all expected reporting months are present.

### Important methodological consideration
The `monthly_reporting_period` field is stored as a character value in the source data (for example, `012018` and `032026`). Therefore, chronological analysis must use an explicitly parsed date rather than relying on string ordering.

This validation is performed across the complete processed dataset using DuckDB so that the large dataset does not need to be loaded into memory as a single pandas DataFrame.

No data is modified in this step.

In [4]:
# ================================================================
# CELL 3 — RAW DATA STRUCTURAL VALIDATION
# ================================================================

print("=" * 70)
print("RAW DATA STRUCTURAL VALIDATION")
print("=" * 70)

structural_query = f"""
WITH raw AS (
    SELECT
        loan_id,
        monthly_reporting_period,
        STRPTIME(
            monthly_reporting_period,
            '%m%Y'
        ) AS reporting_date
    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )
)

SELECT
    COUNT(*) AS total_observations,
    COUNT(DISTINCT loan_id) AS unique_loans,
    COUNT(DISTINCT monthly_reporting_period) AS distinct_reporting_months,
    MIN(reporting_date) AS earliest_reporting_date,
    MAX(reporting_date) AS latest_reporting_date
FROM raw
"""

structural_df = con.execute(
    structural_query
).fetchdf()

display(structural_df)

RAW DATA STRUCTURAL VALIDATION


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_observations,unique_loans,distinct_reporting_months,earliest_reporting_date,latest_reporting_date
0,23563399,458830,99,2018-01-01,2026-03-01


# 4. Monthly Observation Coverage

### Objective
This section examines the number of loan observations available in each reporting month across the complete dataset.

### Why this is necessary
A longitudinal credit-risk model depends on reliable temporal coverage. Examining the number of observations by month helps identify:

- missing reporting months;
- unexpected breaks in the dataset;
- changes in the number of active observations over time;
- the availability of future observations required to construct the six-month early-warning target.

The reporting period is explicitly parsed into a calendar date so that months are ordered chronologically rather than lexicographically.

In [5]:
# ================================================================
# CELL 4 — MONTHLY OBSERVATION COVERAGE
# ================================================================

monthly_coverage_query = f"""
SELECT
    monthly_reporting_period,
    STRPTIME(
        monthly_reporting_period,
        '%m%Y'
    ) AS reporting_date,
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
GROUP BY
    monthly_reporting_period,
    reporting_date
ORDER BY
    reporting_date
"""

monthly_coverage = con.execute(
    monthly_coverage_query
).fetchdf()

print("=" * 70)
print("MONTHLY OBSERVATION COVERAGE")
print("=" * 70)

display(monthly_coverage)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

MONTHLY OBSERVATION COVERAGE


,monthly_reporting_period,reporting_date,observations,unique_loans
0,012018,2018-01-01,165763,165763
1,022018,2018-02-01,316950,316950
2,032018,2018-03-01,458603,458603
3,042018,2018-04-01,458008,458008
4,052018,2018-05-01,457061,457061
...,...,...,...,...
94,112025,2025-11-01,117063,117063
95,122025,2025-12-01,116438,116438
96,012026,2026-01-01,115727,115727
97,022026,2026-02-01,115060,115060


# 5. Loan-Month Uniqueness Validation

### Objective
This section verifies whether each mortgage loan has a unique observation for each reporting month.

### Why this is necessary
The proposed early-warning system treats the dataset as a monthly longitudinal panel in which each row represents one loan observed at one point in time.

Therefore, the combination of:

- `loan_id`
- `monthly_reporting_period`

should uniquely identify an observation.

Duplicate loan-month records could cause incorrect temporal ordering, duplicate observations, and errors when constructing the six-month future outcome. This check therefore establishes the integrity of the panel structure before target generation.

No records are removed at this stage. The purpose is only to identify and quantify any violations.

In [6]:
# ================================================================
# CELL 5 — LOAN-MONTH UNIQUENESS VALIDATION
# ================================================================

duplicate_query = f"""
SELECT
    loan_id,
    monthly_reporting_period,
    COUNT(*) AS observation_count
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
GROUP BY
    loan_id,
    monthly_reporting_period
HAVING COUNT(*) > 1
ORDER BY observation_count DESC
"""

duplicate_df = con.execute(
    duplicate_query
).fetchdf()

print("=" * 70)
print("LOAN-MONTH UNIQUENESS VALIDATION")
print("=" * 70)

print(
    f"Number of duplicated loan-month combinations: "
    f"{len(duplicate_df):,}"
)

if len(duplicate_df) == 0:
    print("STATUS: PASS — Every loan-month combination is unique.")
else:
    print("STATUS: REVIEW REQUIRED — Duplicate loan-month combinations found.")
    display(duplicate_df.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LOAN-MONTH UNIQUENESS VALIDATION
Number of duplicated loan-month combinations: 0
STATUS: PASS — Every loan-month combination is unique.


# 6. Loan-Level Monthly Continuity Validation

### Objective
This section evaluates whether individual mortgage loans have continuous monthly reporting observations throughout their observed performance history.

### Why this is necessary
The early-warning target is defined over six consecutive future calendar months. Therefore, simply selecting the next six rows for a loan is not sufficient.

For example, if a loan has observations in January, February, March, and May, then May must not be treated as the fourth consecutive future month after January.

This validation identifies gaps in individual loan histories so that the target-generation procedure can explicitly require:

`t+1, t+2, t+3, t+4, t+5, t+6`

to be the actual consecutive calendar months.

### Method
For each loan, observations are ordered chronologically and the difference between consecutive reporting dates is calculated.

A one-month difference represents a continuous monthly observation. Differences greater than one month indicate a gap in the loan's reporting history.

No records are removed in this step. The results are used to determine the correct methodology for constructing the forward-looking target.

In [7]:
# ================================================================
# CELL 6 — LOAN-LEVEL MONTHLY CONTINUITY VALIDATION
# ================================================================

continuity_query = f"""
WITH loan_dates AS (
    SELECT
        loan_id,
        STRPTIME(
            monthly_reporting_period,
            '%m%Y'
        ) AS reporting_date
    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )
),

sequenced AS (
    SELECT
        loan_id,
        reporting_date,
        LAG(reporting_date) OVER (
            PARTITION BY loan_id
            ORDER BY reporting_date
        ) AS previous_reporting_date
    FROM loan_dates
),

gaps AS (
    SELECT
        loan_id,
        reporting_date,
        previous_reporting_date,
        DATE_DIFF(
            'month',
            previous_reporting_date,
            reporting_date
        ) AS months_since_previous
    FROM sequenced
    WHERE previous_reporting_date IS NOT NULL
)

SELECT
    COUNT(*) AS consecutive_month_pairs,
    COUNT(*) FILTER (
        WHERE months_since_previous = 1
    ) AS one_month_intervals,
    COUNT(*) FILTER (
        WHERE months_since_previous > 1
    ) AS gap_intervals,
    MAX(months_since_previous) AS maximum_month_gap
FROM gaps
"""

continuity_df = con.execute(
    continuity_query
).fetchdf()

print("=" * 70)
print("LOAN-LEVEL MONTHLY CONTINUITY VALIDATION")
print("=" * 70)

display(continuity_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LOAN-LEVEL MONTHLY CONTINUITY VALIDATION


,consecutive_month_pairs,one_month_intervals,gap_intervals,maximum_month_gap
0,23104569,23104566,3,8


# 7. Inspection of Loan-Level Reporting Gaps

### Objective
This section identifies the specific loan histories containing gaps in monthly reporting.

### Why this is necessary
The previous validation identified three intervals in which consecutive observations for the same loan were separated by more than one calendar month.

Although the number of gaps is extremely small relative to the full dataset, their exact location and duration must be examined before constructing the six-month early-warning target.

This ensures that the target-generation methodology is based on observed calendar months rather than assumptions about row positions.

### Decision rule
Any observation will be considered to have a valid six-month future window only when all six required calendar months are explicitly present.

Therefore, the presence of these gaps will not compromise the target construction.

In [8]:
# ================================================================
# CELL 7 — IDENTIFY REPORTING GAPS
# ================================================================

gap_query = f"""
WITH loan_dates AS (
    SELECT
        loan_id,
        STRPTIME(
            monthly_reporting_period,
            '%m%Y'
        ) AS reporting_date
    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )
),

sequenced AS (
    SELECT
        loan_id,
        reporting_date,
        LAG(reporting_date) OVER (
            PARTITION BY loan_id
            ORDER BY reporting_date
        ) AS previous_reporting_date
    FROM loan_dates
)

SELECT
    loan_id,
    previous_reporting_date,
    reporting_date,
    DATE_DIFF(
        'month',
        previous_reporting_date,
        reporting_date
    ) AS months_since_previous
FROM sequenced
WHERE previous_reporting_date IS NOT NULL
  AND DATE_DIFF(
        'month',
        previous_reporting_date,
        reporting_date
      ) > 1
ORDER BY
    loan_id,
    previous_reporting_date
"""

gap_df = con.execute(
    gap_query
).fetchdf()

print("=" * 70)
print("IDENTIFIED REPORTING GAPS")
print("=" * 70)

print(f"Number of gap records: {len(gap_df):,}")

display(gap_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

IDENTIFIED REPORTING GAPS
Number of gap records: 3


,loan_id,previous_reporting_date,reporting_date,months_since_previous
0,106744050927,2019-04-01,2019-12-01,8
1,320025628803,2019-09-01,2019-12-01,3
2,474263955324,2018-02-01,2018-05-01,3


# 8. Current Delinquency Status Validation

### Objective
This section examines the distribution and validity of the `current_delinquency_status` field, which is the key variable used to define the six-month early-warning outcome.

### Why this is necessary
The proposed early-warning target identifies loans that are currently performing but subsequently become 30 or more days delinquent.

Therefore, we need to establish the observed delinquency-status codes before defining the target.

For the target construction:

- `00` represents a currently performing loan and defines the eligible population at month *t*.
- Status codes `01` and above represent delinquency of at least 30 days and therefore constitute an adverse future event.
- `XX` represents an unknown status and cannot be reliably used to determine whether a future delinquency event occurred.

### Method
The complete processed dataset is scanned to obtain the frequency of each observed delinquency-status code.

No records are modified or excluded in this step. The results will be used to formally define the target-generation rules in the next stage.

In [9]:
# ================================================================
# CELL 8 — CURRENT DELINQUENCY STATUS VALIDATION
# ================================================================

delinquency_query = f"""
SELECT
    current_delinquency_status,
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
GROUP BY
    current_delinquency_status
ORDER BY
    CASE
        WHEN current_delinquency_status = 'XX' THEN 999
        ELSE TRY_CAST(
            current_delinquency_status AS INTEGER
        )
    END
"""

delinquency_df = con.execute(
    delinquency_query
).fetchdf()

print("=" * 70)
print("CURRENT DELINQUENCY STATUS DISTRIBUTION")
print("=" * 70)

display(delinquency_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CURRENT DELINQUENCY STATUS DISTRIBUTION


,current_delinquency_status,observations,unique_loans
0,00,22823077,458829
1,01,248286,72690
2,02,84999,37968
3,03,51475,30251
4,04,37821,25053
...,...,...,...
85,85,2,2
86,86,2,2
87,87,2,2
88,88,2,2


# 9. Delinquency Status Code Validation

### Objective
This section validates the coding structure of the current delinquency status field before using it to construct the early-warning target.

### Why this is necessary
The target requires a clear distinction between:

1. a currently performing loan;
2. a future delinquency event; and
3. an unknown future status.

The target methodology therefore uses the following rules:

- `00` → currently current;
- numeric status `>= 01` → at least 30 days delinquent;
- `XX` → unknown and therefore unsuitable for determining a future outcome.

The analysis does not impose an arbitrary upper delinquency threshold. Any numeric delinquency status from `01` onward represents an adverse event for the purpose of the six-month early-warning target.

This validation checks whether any status values fall outside the expected numeric codes and `XX`.

In [10]:
# ================================================================
# CELL 9 — DELINQUENCY STATUS CODE VALIDATION
# ================================================================

status_validation_query = f"""
WITH statuses AS (
    SELECT DISTINCT
        current_delinquency_status
    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )
)

SELECT
    current_delinquency_status,
    TRY_CAST(
        current_delinquency_status AS INTEGER
    ) AS numeric_status,
    CASE
        WHEN current_delinquency_status = 'XX'
            THEN 'UNKNOWN'
        WHEN TRY_CAST(
            current_delinquency_status AS INTEGER
        ) = 0
            THEN 'CURRENT'
        WHEN TRY_CAST(
            current_delinquency_status AS INTEGER
        ) >= 1
            THEN 'DELINQUENT'
        ELSE 'UNEXPECTED'
    END AS status_category
FROM statuses
ORDER BY
    CASE
        WHEN current_delinquency_status = 'XX'
            THEN 999
        ELSE TRY_CAST(
            current_delinquency_status AS INTEGER
        )
    END
"""

status_validation_df = con.execute(
    status_validation_query
).fetchdf()

print("=" * 70)
print("DELINQUENCY STATUS CODE VALIDATION")
print("=" * 70)

display(status_validation_df)

unexpected_statuses = status_validation_df[
    status_validation_df["status_category"] == "UNEXPECTED"
]

print(
    f"\nUnexpected status codes: "
    f"{len(unexpected_statuses)}"
)

if len(unexpected_statuses) == 0:
    print(
        "STATUS: PASS — All status codes are either "
        "numeric delinquency codes or XX."
    )
else:
    print(
        "STATUS: REVIEW REQUIRED — Unexpected status codes found."
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DELINQUENCY STATUS CODE VALIDATION


,current_delinquency_status,numeric_status,status_category
0,00,0,CURRENT
1,01,1,DELINQUENT
2,02,2,DELINQUENT
3,03,3,DELINQUENT
4,04,4,DELINQUENT
...,...,...,...
85,85,85,DELINQUENT
86,86,86,DELINQUENT
87,87,87,DELINQUENT
88,88,88,DELINQUENT



Unexpected status codes: 0
STATUS: PASS — All status codes are either numeric delinquency codes or XX.


# 10. Unknown Delinquency Status and Missingness Validation

### Objective
This section examines the occurrence of unknown delinquency statuses (`XX`) and missing delinquency values across the reporting timeline.

### Why this is necessary
The six-month early-warning target requires a reliable determination of whether a currently performing loan becomes delinquent during the following six calendar months.

An `XX` status indicates that the delinquency status is unknown. Therefore, it must not be interpreted as a current or non-delinquent observation.

Similarly, a missing monthly observation cannot automatically be treated as a month with no delinquency. It represents an absence of observed information.

The target-generation procedure will therefore distinguish between:

- observed current status (`00`);
- observed delinquency (`01` or higher);
- unknown status (`XX`);
- unavailable/missing future observations.

This validation helps quantify the extent of unknown status information before the target is constructed.

In [11]:
# ================================================================
# CELL 10 — UNKNOWN STATUS AND MISSINGNESS VALIDATION
# ================================================================

unknown_query = f"""
SELECT
    STRPTIME(
        monthly_reporting_period,
        '%m%Y'
    ) AS reporting_date,

    COUNT(*) AS total_observations,

    COUNT(*) FILTER (
        WHERE current_delinquency_status = '00'
    ) AS current_observations,

    COUNT(*) FILTER (
        WHERE current_delinquency_status = 'XX'
    ) AS unknown_observations,

    COUNT(*) FILTER (
        WHERE current_delinquency_status IS NULL
           OR TRIM(current_delinquency_status) = ''
    ) AS missing_status_observations

FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)

GROUP BY reporting_date

ORDER BY reporting_date
"""

unknown_df = con.execute(
    unknown_query
).fetchdf()

print("=" * 70)
print("UNKNOWN / MISSING DELINQUENCY STATUS BY MONTH")
print("=" * 70)

display(unknown_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

UNKNOWN / MISSING DELINQUENCY STATUS BY MONTH


,reporting_date,total_observations,current_observations,unknown_observations,missing_status_observations
0,2018-01-01,165763,165661,0,0
1,2018-02-01,316950,315656,226,0
2,2018-03-01,458603,456608,595,0
3,2018-04-01,458008,455361,948,0
4,2018-05-01,457061,453967,1394,0
...,...,...,...,...,...
94,2025-11-01,117063,113208,0,0
95,2025-12-01,116438,112897,0,0
96,2026-01-01,115727,112194,0,0
97,2026-02-01,115060,111554,0,0


# 11. Overall Missing-Value Validation

### Objective
This section measures the amount of missing information in each variable across the complete processed dataset.

### Why this is necessary
Missing-value treatment is an important part of a reliable credit-risk modelling pipeline. Different variables may have different reasons for being unavailable, and applying a blanket imputation rule without first understanding the missingness pattern could introduce bias.

Therefore, before feature engineering, we quantify:

- total missing observations;
- percentage of observations missing;
- the distribution of missingness across the available variables.

This assessment will guide the later data-cleaning and feature-engineering decisions.

### Method
The complete dataset is scanned using DuckDB. Missingness is measured using NULL values detected by the data-processing engine.

No imputation, deletion, or transformation is performed at this stage.

In [13]:
# ================================================================
# CELL 11 — OVERALL MISSING-VALUE VALIDATION
# ================================================================

# Get the column names from the validated schema
raw_columns = schema_df["column_name"].tolist()

missing_expressions = []

for col in raw_columns:
    missing_expressions.append(
        f"""
        COUNT(*) - COUNT("{col}") AS "{col}"
        """
    )

missing_query = f"""
SELECT
    COUNT(*) AS total_observations,
    {",".join(missing_expressions)}
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
"""

missing_wide = con.execute(
    missing_query
).fetchdf()

# ------------------------------------------------
# Convert wide missingness output to readable table
# ------------------------------------------------

total_observations = int(
    missing_wide.loc[0, "total_observations"]
)

missing_records = []

for col in raw_columns:
    missing_count = int(
        missing_wide.loc[0, col]
    )

    missing_pct = (
        missing_count /
        total_observations *
        100
    )

    missing_records.append({
        "column_name": col,
        "missing_observations": missing_count,
        "missing_percentage": missing_pct
    })

missing_df = pd.DataFrame(
    missing_records
).sort_values(
    "missing_percentage",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("OVERALL MISSING-VALUE VALIDATION")
print("=" * 70)

print(
    f"Total observations checked: "
    f"{total_observations:,}"
)

display(missing_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OVERALL MISSING-VALUE VALIDATION
Total observations checked: 23,563,399


,column_name,missing_observations,missing_percentage
0,interest_only_first_pi_date,23563399,100.000000
1,mi_cancellation_indicator,23563399,100.000000
2,months_to_amortization,23563399,100.000000
3,zero_balance_code,23218289,98.535398
4,zero_balance_effective_date,23218289,98.535398
5,mortgage_insurance_percentage,16622586,70.544093
6,co_borrower_credit_score,13410280,56.911484
7,loan_payment_history,11464285,48.652934
8,remaining_months_maturity,568306,2.411817
9,servicer_name,345110,1.464602


# 12. Blank-String Validation

### Objective
This section checks whether character-based variables contain empty or whitespace-only values in addition to database NULL values.

### Why this is necessary
Missing information in a large CSV dataset can be represented in more than one way. A value may be stored as:

- NULL;
- an empty string;
- whitespace;
- a valid categorical value.

The previous analysis measured NULL values only. This step verifies whether additional missingness is encoded as blank character strings.

This distinction is important because blank strings should be treated as missing during later data preparation, whereas legitimate categorical values should be preserved.

No values are modified in this step.

In [14]:
# ================================================================
# CELL 12 — BLANK-STRING VALIDATION
# ================================================================

varchar_columns = schema_df.loc[
    schema_df["column_type"].str.upper().str.contains("VARCHAR"),
    "column_name"
].tolist()

blank_expressions = []

for col in varchar_columns:
    blank_expressions.append(
        f"""
        COUNT(*) FILTER (
            WHERE "{col}" IS NOT NULL
              AND TRIM("{col}") = ''
        ) AS "{col}"
        """
    )

blank_query = f"""
SELECT
    COUNT(*) AS total_observations,
    {",".join(blank_expressions)}
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
"""

blank_wide = con.execute(
    blank_query
).fetchdf()

blank_records = []

for col in varchar_columns:
    blank_count = int(
        blank_wide.loc[0, col]
    )

    blank_pct = (
        blank_count /
        total_observations *
        100
    )

    blank_records.append({
        "column_name": col,
        "blank_observations": blank_count,
        "blank_percentage": blank_pct
    })

blank_df = pd.DataFrame(
    blank_records
).sort_values(
    "blank_percentage",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("BLANK-STRING VALIDATION")
print("=" * 70)

display(blank_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BLANK-STRING VALIDATION


,column_name,blank_observations,blank_percentage
0,monthly_reporting_period,0,0.0
1,channel,0,0.0
2,seller_name,0,0.0
3,servicer_name,0,0.0
4,origination_date,0,0.0
5,first_payment_date,0,0.0
6,maturity_date,0,0.0
7,first_time_home_buyer,0,0.0
8,loan_purpose,0,0.0
9,property_type,0,0.0


# 13. Variable Classification for Modelling

### Objective
This section classifies the 42 source variables according to their role in the proposed credit-risk early-warning framework.

### Why this is necessary
Not every variable in the source dataset should be used as a model predictor. Some variables identify observations, some define the target, some contain information that is only available after a credit event, and others are useful predictors.

A formal variable classification prevents accidental target leakage and provides a transparent basis for feature selection.

### Classification principles

**1. Identifiers**
Variables used to identify loans or reporting observations but not used as predictive features.

**2. Target / eligibility variables**
Variables required to define whether an observation is eligible for prediction or whether a future adverse event occurred.

**3. Candidate predictors**
Variables containing information that would reasonably be available at the prediction month and may contribute to credit-risk prediction.

**4. Conditionally applicable predictors**
Variables that may legitimately be missing because they apply only to a subset of loans. Their missingness will be investigated before deciding on an appropriate representation.

**5. Excluded variables**
Variables that may contain post-event information, redundant information, or information unsuitable for the early-warning prediction task.

The final predictor set will be determined only after examining data availability and potential leakage.

In [15]:
# ================================================================
# CELL 13 — VARIABLE CLASSIFICATION
# ================================================================

variable_classification = pd.DataFrame({

    "column_name": [
        "loan_id",
        "monthly_reporting_period",
        "channel",
        "seller_name",
        "servicer_name",
        "original_interest_rate",
        "current_interest_rate",
        "original_upb",
        "current_actual_upb",
        "original_loan_term",
        "origination_date",
        "first_payment_date",
        "loan_age",
        "remaining_months_legal_maturity",
        "remaining_months_maturity",
        "maturity_date",
        "original_ltv",
        "original_cltv",
        "number_of_borrowers",
        "dti",
        "borrower_credit_score",
        "co_borrower_credit_score",
        "first_time_home_buyer",
        "loan_purpose",
        "property_type",
        "number_of_units",
        "occupancy_status",
        "property_state",
        "msa",
        "zip_code_short",
        "mortgage_insurance_percentage",
        "amortization_type",
        "prepayment_penalty_indicator",
        "interest_only_indicator",
        "interest_only_first_pi_date",
        "months_to_amortization",
        "current_delinquency_status",
        "loan_payment_history",
        "modification_flag",
        "mi_cancellation_indicator",
        "zero_balance_code",
        "zero_balance_effective_date"
    ],

    "role": [
        "Identifier",
        "Time Index",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Conditionally Applicable Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Conditionally Applicable Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Candidate Predictor",
        "Excluded",
        "Excluded",
        "Target Eligibility Variable",
        "Behavioural Predictor Source",
        "Candidate Predictor",
        "Excluded",
        "Excluded",
        "Excluded"
    ],

    "reason": [
        "Identifies the mortgage loan.",
        "Identifies the observation month.",
        "Loan acquisition/channel characteristic.",
        "Seller-related loan characteristic.",
        "Servicing-related loan characteristic.",
        "Original contractual interest rate.",
        "Current interest rate available at prediction time.",
        "Original loan balance.",
        "Current outstanding loan balance.",
        "Original contractual loan term.",
        "Loan origination timing.",
        "Scheduled first payment timing.",
        "Age of the loan at observation.",
        "Remaining legal maturity.",
        "Remaining scheduled maturity.",
        "Contractual maturity date.",
        "Original loan-to-value ratio.",
        "Original combined loan-to-value ratio.",
        "Number of borrowers.",
        "Debt-to-income ratio.",
        "Borrower credit score at origination.",
        "Co-borrower credit information.",
        "First-time home-buyer indicator.",
        "Purpose of the mortgage.",
        "Property type.",
        "Number of property units.",
        "Occupancy status.",
        "Property state.",
        "Metropolitan statistical area.",
        "Short ZIP-code geography.",
        "Mortgage insurance percentage where applicable.",
        "Amortization structure.",
        "Prepayment penalty indicator.",
        "Interest-only indicator.",
        "Excluded because it is entirely unavailable in this dataset.",
        "Excluded because it is entirely unavailable in this dataset.",
        "Used to determine whether the loan is currently current; not used as a predictor.",
        "Historical payment-performance information used to engineer backward-looking behavioural features.",
        "Loan modification information available at the observation date.",
        "Potentially reflects a later insurance event and is excluded from the initial early-warning predictor set.",
        "Represents a zero-balance event and may occur after the loan exits active performance; excluded from predictors.",
        "Date associated with the zero-balance event; excluded from predictors."
    ]
})

print("=" * 70)
print("VARIABLE CLASSIFICATION")
print("=" * 70)

display(variable_classification)

VARIABLE CLASSIFICATION


,column_name,role,reason
0,loan_id,Identifier,Identifies the mortgage loan.
1,monthly_reporting_period,Time Index,Identifies the observation month.
2,channel,Candidate Predictor,Loan acquisition/channel characteristic.
3,seller_name,Candidate Predictor,Seller-related loan characteristic.
4,servicer_name,Candidate Predictor,Servicing-related loan characteristic.
5,original_interest_rate,Candidate Predictor,Original contractual interest rate.
6,current_interest_rate,Candidate Predictor,Current interest rate available at prediction ...
7,original_upb,Candidate Predictor,Original loan balance.
8,current_actual_upb,Candidate Predictor,Current outstanding loan balance.
9,original_loan_term,Candidate Predictor,Original contractual loan term.


# 14. Numerical Variable Range Validation

### Objective
This section validates the ranges of numerical variables in the raw dataset and identifies values that may be impossible, implausible, or require special treatment.

### Why this is necessary
Missing-value analysis alone cannot establish data quality. A variable may contain no missing values but still contain values outside its expected domain.

For a mortgage credit-risk dataset, examples include:

- credit scores outside plausible ranges;
- negative loan balances;
- invalid loan-to-value ratios;
- negative debt-to-income ratios;
- invalid borrower counts;
- negative loan ages;
- invalid interest rates.

This validation is performed before feature engineering so that later transformations are based on verified source values.

### Method
The complete dataset is summarized using minimum, maximum, and selected distribution statistics for numerical variables.

No observations are removed or modified at this stage. Any potential anomalies identified here will be investigated before deciding on a treatment.

In [16]:
# ================================================================
# CELL 14 — NUMERICAL VARIABLE RANGE VALIDATION
# ================================================================

numeric_columns = schema_df.loc[
    schema_df["column_type"].str.upper().isin(
        ["BIGINT", "INTEGER", "DOUBLE", "FLOAT"]
    ),
    "column_name"
].tolist()

numeric_expressions = []

for col in numeric_columns:
    numeric_expressions.append(
        f"""
        MIN("{col}") AS "{col}__min",
        MAX("{col}") AS "{col}__max",
        AVG("{col}") AS "{col}__mean"
        """
    )

numeric_query = f"""
SELECT
    {",".join(numeric_expressions)}
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
"""

numeric_wide = con.execute(
    numeric_query
).fetchdf()

numeric_records = []

for col in numeric_columns:
    numeric_records.append({
        "column_name": col,
        "minimum": numeric_wide.loc[0, f"{col}__min"],
        "maximum": numeric_wide.loc[0, f"{col}__max"],
        "mean": numeric_wide.loc[0, f"{col}__mean"]
    })

numeric_summary = pd.DataFrame(
    numeric_records
)

print("=" * 70)
print("NUMERICAL VARIABLE RANGE VALIDATION")
print("=" * 70)

display(numeric_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

NUMERICAL VARIABLE RANGE VALIDATION


,column_name,minimum,maximum,mean
0,loan_id,1.000010e+11,9.999988e+11,5.494819e+11
1,original_interest_rate,2.500000e+00,6.250000e+00,4.229162e+00
2,current_interest_rate,2.500000e+00,6.250000e+00,4.222422e+00
3,original_upb,1.000000e+04,1.307000e+06,2.183784e+05
4,current_actual_upb,0.000000e+00,1.299082e+06,1.695489e+05
5,original_loan_term,6.000000e+01,3.600000e+02,3.246253e+02
6,loan_age,-3.000000e+00,1.160000e+02,3.465133e+01
7,remaining_months_legal_maturity,0.000000e+00,4.820000e+02,2.915774e+02
8,remaining_months_maturity,0.000000e+00,4.800000e+02,2.823384e+02
9,original_ltv,3.000000e+00,1.050000e+02,7.435369e+01


# 15. Investigation of Negative Loan Age

### Objective
This section investigates observations where `loan_age` is negative.

### Why this is necessary
The numerical range validation identified a minimum loan age of -3 months. A negative value may initially appear anomalous, but source-specific definitions must be considered before modifying the data.

Fannie Mae defines Loan Age as the number of calendar months since the mortgage loan's origination date, with origination based on the first full month of interest accrual. Therefore, negative loan-age values should be investigated against the loan's origination date, first payment date, and reporting month rather than automatically treated as errors.

### Validation approach
For observations with negative loan age, we compare:

- monthly reporting period;
- origination date;
- first payment date;
- loan age.

The purpose is to determine whether the negative values follow the source dataset's documented timing conventions.

No values are changed in this step.

In [17]:
# ================================================================
# CELL 15 — INVESTIGATE NEGATIVE LOAN AGE
# ================================================================

negative_age_query = f"""
SELECT
    loan_id,
    monthly_reporting_period,
    origination_date,
    first_payment_date,
    loan_age,
    current_delinquency_status
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
WHERE loan_age < 0
ORDER BY
    loan_id,
    monthly_reporting_period
LIMIT 100
"""

negative_age_df = con.execute(
    negative_age_query
).fetchdf()

negative_age_count = con.execute(
    f"""
    SELECT COUNT(*) AS negative_loan_age_observations
    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )
    WHERE loan_age < 0
    """
).fetchdf()

print("=" * 70)
print("NEGATIVE LOAN AGE INVESTIGATION")
print("=" * 70)

print(
    "Total observations with loan_age < 0:",
    int(
        negative_age_count.loc[
            0,
            "negative_loan_age_observations"
        ]
    )
)

display(negative_age_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

NEGATIVE LOAN AGE INVESTIGATION
Total observations with loan_age < 0: 77500


,loan_id,monthly_reporting_period,origination_date,first_payment_date,loan_age,current_delinquency_status
0,100010739040,022018,022018,042018,-1.0,00
1,100031627426,032018,032018,052018,-1.0,00
2,100035132875,022018,022018,042018,-1.0,00
3,100042898191,012018,012018,032018,-1.0,00
4,100092212189,022018,022018,042018,-1.0,00
...,...,...,...,...,...,...
95,101066715823,022018,022018,042018,-1.0,00
96,101105462920,012018,012018,032018,-1.0,00
97,101137059305,012018,012018,032018,-1.0,00
98,101190413434,012018,012018,032018,-1.0,00


# 16. Date Field Parsing and Consistency Validation

### Objective
This section validates the date-related variables used in the longitudinal credit-risk analysis.

### Why this is necessary
The early-warning framework depends on accurate temporal ordering. The reporting month, origination date, first payment date, and maturity date must therefore be interpretable as valid calendar dates.

The following relationships are examined:

- `monthly_reporting_period` → reporting month
- `origination_date` → loan origination timing
- `first_payment_date` → first scheduled payment timing
- `maturity_date` → contractual maturity timing

The validation checks:
1. whether the date fields can be successfully parsed;
2. whether any dates are invalid or missing;
3. whether first payment occurs before origination;
4. whether maturity occurs before origination;
5. whether reporting observations occur within a plausible relationship to the loan's lifecycle.

No values are modified in this step.

### Important methodological principle
The source dataset contains legitimate timing conventions around origination and first payment. Therefore, apparent timing differences will be investigated rather than automatically treated as data errors.

In [18]:
# Parse and validate the major date fields.
# No source values are modified.

date_validation_query = f"""
WITH parsed AS (
    SELECT
        loan_id,
        monthly_reporting_period,

        TRY_STRPTIME(
            monthly_reporting_period,
            '%m%Y'
        ) AS reporting_date,

        TRY_STRPTIME(
            origination_date,
            '%m%Y'
        ) AS origination_date_parsed,

        TRY_STRPTIME(
            first_payment_date,
            '%m%Y'
        ) AS first_payment_date_parsed,

        TRY_STRPTIME(
            maturity_date,
            '%m%Y'
        ) AS maturity_date_parsed

    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )
)

SELECT
    COUNT(*) AS total_observations,

    COUNT(*) FILTER (
        WHERE reporting_date IS NULL
    ) AS invalid_reporting_dates,

    COUNT(*) FILTER (
        WHERE origination_date_parsed IS NULL
    ) AS missing_or_invalid_origination_dates,

    COUNT(*) FILTER (
        WHERE first_payment_date_parsed IS NULL
    ) AS missing_or_invalid_first_payment_dates,

    COUNT(*) FILTER (
        WHERE maturity_date_parsed IS NULL
    ) AS missing_or_invalid_maturity_dates,

    COUNT(*) FILTER (
        WHERE origination_date_parsed IS NOT NULL
          AND first_payment_date_parsed IS NOT NULL
          AND first_payment_date_parsed < origination_date_parsed
    ) AS first_payment_before_origination,

    COUNT(*) FILTER (
        WHERE origination_date_parsed IS NOT NULL
          AND maturity_date_parsed IS NOT NULL
          AND maturity_date_parsed < origination_date_parsed
    ) AS maturity_before_origination,

    COUNT(*) FILTER (
        WHERE reporting_date IS NOT NULL
          AND origination_date_parsed IS NOT NULL
          AND reporting_date < origination_date_parsed
    ) AS reporting_before_origination

FROM parsed
"""

date_validation_df = con.execute(
    date_validation_query
).fetchdf()

print("=" * 70)
print("DATE FIELD PARSING AND CONSISTENCY VALIDATION")
print("=" * 70)

display(date_validation_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DATE FIELD PARSING AND CONSISTENCY VALIDATION


,total_observations,invalid_reporting_dates,missing_or_invalid_origination_dates,missing_or_invalid_first_payment_dates,missing_or_invalid_maturity_dates,first_payment_before_origination,maturity_before_origination,reporting_before_origination
0,23563399,0,0,0,345110,0,9,3


# 17. Investigation of Date-Timing Anomalies

### Objective
This section investigates the small number of observations identified with apparently inconsistent date relationships.

### Why this is necessary
The previous validation identified:

- 9 observations where maturity date precedes origination date;
- 3 observations where the reporting month precedes the origination date.

These records represent a very small fraction of the full dataset, but they must be examined before deciding whether they are source-defined timing conventions or genuine data-quality anomalies.

### Validation approach
The affected records are displayed with:

- loan identifier;
- reporting month;
- origination date;
- first payment date;
- maturity date;
- loan age;
- original loan term.

The purpose is diagnostic only. No observations are deleted or modified.

### Decision principle
A source-defined timing convention will be retained. A genuinely inconsistent record will be excluded only if there is sufficient evidence that the value cannot be interpreted reliably for the modelling task.

In [19]:
date_anomaly_query = f"""
WITH parsed AS (
    SELECT
        loan_id,
        monthly_reporting_period,

        TRY_STRPTIME(
            monthly_reporting_period,
            '%m%Y'
        ) AS reporting_date,

        TRY_STRPTIME(
            origination_date,
            '%m%Y'
        ) AS origination_date_parsed,

        TRY_STRPTIME(
            first_payment_date,
            '%m%Y'
        ) AS first_payment_date_parsed,

        TRY_STRPTIME(
            maturity_date,
            '%m%Y'
        ) AS maturity_date_parsed,

        loan_age,
        original_loan_term,
        current_delinquency_status

    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )
)

SELECT
    loan_id,
    monthly_reporting_period,
    origination_date_parsed AS origination_date,
    first_payment_date_parsed AS first_payment_date,
    maturity_date_parsed AS maturity_date,
    loan_age,
    original_loan_term,
    current_delinquency_status,

    CASE
        WHEN reporting_date < origination_date_parsed
            THEN 'REPORTING_BEFORE_ORIGINATION'
        WHEN maturity_date_parsed < origination_date_parsed
            THEN 'MATURITY_BEFORE_ORIGINATION'
    END AS anomaly_type

FROM parsed

WHERE
    (
        reporting_date < origination_date_parsed
    )
    OR
    (
        maturity_date_parsed < origination_date_parsed
    )

ORDER BY
    anomaly_type,
    loan_id,
    reporting_date
"""

date_anomalies_df = con.execute(
    date_anomaly_query
).fetchdf()

print("=" * 70)
print("DATE-TIMING ANOMALY INVESTIGATION")
print("=" * 70)

print(f"Total anomaly observations: {len(date_anomalies_df):,}")

display(date_anomalies_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DATE-TIMING ANOMALY INVESTIGATION
Total anomaly observations: 12


,loan_id,monthly_reporting_period,origination_date,first_payment_date,maturity_date,loan_age,original_loan_term,current_delinquency_status,anomaly_type
0,409954531803,072019,2017-12-01,2018-02-01,1959-06-01,18.0,360,00,MATURITY_BEFORE_ORIGINATION
1,409954531803,082019,2017-12-01,2018-02-01,1959-06-01,19.0,360,00,MATURITY_BEFORE_ORIGINATION
2,409954531803,092019,2017-12-01,2018-02-01,1959-06-01,20.0,360,00,MATURITY_BEFORE_ORIGINATION
3,409954531803,102019,2017-12-01,2018-02-01,1959-06-01,21.0,360,00,MATURITY_BEFORE_ORIGINATION
4,409954531803,112019,2017-12-01,2018-02-01,1959-06-01,22.0,360,00,MATURITY_BEFORE_ORIGINATION
5,409954531803,122019,2017-12-01,2018-02-01,1959-06-01,23.0,360,00,MATURITY_BEFORE_ORIGINATION
6,409954531803,012020,2017-12-01,2018-02-01,1959-06-01,24.0,360,00,MATURITY_BEFORE_ORIGINATION
7,409954531803,022020,2017-12-01,2018-02-01,1959-06-01,25.0,360,00,MATURITY_BEFORE_ORIGINATION
8,409954531803,032020,2017-12-01,2018-02-01,1959-06-01,26.0,360,00,MATURITY_BEFORE_ORIGINATION
9,611677671603,012018,2018-03-01,2018-05-01,2047-06-01,-3.0,350,00,REPORTING_BEFORE_ORIGINATION


# 18. Categorical and Flag Variable Validation

### Objective
This section validates the categorical and indicator variables in the Fannie Mae loan performance dataset.

### Why this is necessary
Categorical variables are important potential predictors in the credit-risk model, but they must be checked before feature engineering.

The validation examines:

- distinct category counts;
- observed category values;
- unexpected or unusual codes;
- indicator variables and their valid states;
- high-cardinality variables that may require special encoding.

Particular attention is given to variables such as:
- channel;
- loan purpose;
- property type;
- occupancy status;
- first-time home buyer indicator;
- amortization type;
- modification flag;
- prepayment penalty indicator;
- interest-only indicator;
- property state.

High-cardinality variables such as seller, servicer, MSA, and ZIP-code information will be evaluated separately because directly one-hot encoding every category may create an unnecessarily large feature space.

No values are modified in this step.

In [20]:
# Define categorical/flag variables that require validation.
categorical_columns = [
    "channel",
    "seller_name",
    "servicer_name",
    "first_time_home_buyer",
    "loan_purpose",
    "property_type",
    "occupancy_status",
    "property_state",
    "msa",
    "zip_code_short",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_indicator",
    "modification_flag"
]

categorical_summary_records = []

for col in categorical_columns:
    query = f"""
    SELECT
        COUNT(DISTINCT "{col}") AS distinct_values,
        COUNT(*) FILTER (
            WHERE "{col}" IS NULL
        ) AS null_values
    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )
    """

    result = con.execute(query).fetchdf().iloc[0]

    categorical_summary_records.append({
        "column_name": col,
        "distinct_values": int(result["distinct_values"]),
        "null_values": int(result["null_values"])
    })

categorical_summary_df = pd.DataFrame(
    categorical_summary_records
).sort_values(
    "distinct_values",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("CATEGORICAL VARIABLE SUMMARY")
print("=" * 70)

display(categorical_summary_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (6 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CATEGORICAL VARIABLE SUMMARY


,column_name,distinct_values,null_values
0,zip_code_short,889,123
1,msa,448,0
2,servicer_name,87,345110
3,property_state,54,0
4,seller_name,49,0
5,property_type,5,0
6,channel,3,0
7,loan_purpose,3,0
8,occupancy_status,3,0
9,first_time_home_buyer,2,26


# 19. Low-Cardinality Category Code Validation

### Objective
This section examines the actual observed category values for the low-cardinality categorical and indicator variables.

### Why this is necessary
The previous validation established the number of distinct values but did not show what those values represent.

Before encoding categorical variables, we need to verify that:
- indicator variables contain expected codes;
- categorical variables contain valid source categories;
- missing values are distinguishable from legitimate categories;
- constant variables can be identified for exclusion.

### Variables examined
The following low-cardinality variables are inspected:

- channel
- first-time home buyer indicator
- loan purpose
- property type
- occupancy status
- amortization type
- prepayment penalty indicator
- interest-only indicator
- modification flag

High-cardinality variables such as seller, servicer, MSA, and ZIP code are not expanded here because their encoding strategy will be considered separately.

No values are modified in this step.

In [21]:
# Inspect the actual values of all low-cardinality categorical variables
# in a single DuckDB scan.

low_cardinality_columns = [
    "channel",
    "first_time_home_buyer",
    "loan_purpose",
    "property_type",
    "occupancy_status",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_indicator",
    "modification_flag"
]

value_expressions = []

for col in low_cardinality_columns:
    value_expressions.append(
        f'list_sort(list(DISTINCT "{col}")) AS "{col}"'
    )

low_cardinality_query = f"""
SELECT
    {",".join(value_expressions)}
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
"""

low_cardinality_values = con.execute(
    low_cardinality_query
).fetchdf()

print("=" * 70)
print("LOW-CARDINALITY CATEGORY VALUES")
print("=" * 70)

for col in low_cardinality_columns:
    values = low_cardinality_values.loc[0, col]

    print(f"\n{col}")
    print("-" * len(col))

    if values is None:
        print("No observed values")
    else:
        print(values)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LOW-CARDINALITY CATEGORY VALUES

channel
-------
['B' 'C' 'R']

first_time_home_buyer
---------------------
['N' 'Y' None]

loan_purpose
------------
['C' 'P' 'R']

property_type
-------------
['CO' 'CP' 'MH' 'PU' 'SF']

occupancy_status
----------------
['I' 'P' 'S']

amortization_type
-----------------
['FRM']

prepayment_penalty_indicator
----------------------------
['N']

interest_only_indicator
-----------------------
['N']

modification_flag
-----------------
['N' 'Y' None]
